In [7]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

# STEP 1: LOAD RIN-CORRECTED EXPRESSION MATRIX
print("=== STEP 1: Loading RIN-Corrected FPKM Matrix ===")
fpkm_file = "GSE104687_RIN_corrected_FPKM.csv"

# Load the pre-corrected normalized expression data
fpkm_norm = pd.read_csv(fpkm_file)
fpkm_norm = fpkm_norm.rename(columns={fpkm_norm.columns[0]: "gene_id"})

print(
    f"Loaded expression matrix: {fpkm_norm.shape[0]} Genes x {fpkm_norm.shape[1] - 1} Samples"
)


# STEP 2: LOAD SAMPLE METADATA & DONOR INFORMATION
print("\n=== STEP 2: Loading Sample and Donor Metadata ===")
samples_df = pd.read_csv("columns-samples.csv")
donor_df = pd.read_csv("DonorInformation-2.csv")

# Define and apply primary group classification function
def set_primary_group(row):
    nia = row.get("nia_reagan")
    dementia = row.get("act_demented")
    if nia in [2, 3] and dementia == "No Dementia":
        return "Resilience-like"
    elif nia in [2, 3] and dementia == "Dementia":
        return "Pathology-positive dementia"
    elif nia in [0, 1] and dementia == "No Dementia":
        return "Low-pathology reference"
    elif nia in [0, 1] and dementia == "Dementia":
        return "Low-pathology dementia"
    return None

donor_df["primary_group"] = donor_df.apply(set_primary_group, axis=1)

# Filter for Hippocampal tissue samples
hip_samples = samples_df[samples_df["structure_acronym"] == "HIP"].copy()

print(f"Total Hippocampal Samples Identified: {len(hip_samples)}")
print(f"Unique Donors in HIP: {hip_samples['donor_id'].nunique()}")


# STEP 3: MERGE DONOR PATHOLOGY METADATA & FILTER COHORT
print("\n=== STEP 3: Merging Donor Variables & Filtering Cohort ===")

hip_merged = hip_samples.merge(donor_df, on="donor_id", how="left")
group_col = "primary_group"

target_groups = ["Resilience-like", "Pathology-positive dementia"]
hip_target = hip_merged[
    hip_merged[group_col].isin(target_groups)
].copy()

counts = hip_target[group_col].value_counts()
pathology_count = counts.get("Pathology-positive dementia", 0)
resilience_count = counts.get("Resilience-like", 0)
group_missingness = hip_target[group_col].isna().sum()

print("Cohort Sample Counts by Group:")
print(counts)


# STEP 3.5: Logging RIN QC Statement
print("\n=== STEP 3.5: Logging RIN QC Statement ===")

qc_statement = """=== HIPPOCAMPAL RNA-SEQ QUALITY CONTROL (RIN) STATEMENT ===
Expression Matrix Used: GSE104687_RIN_corrected_FPKM.csv.gz
RIN Adjustment Status: Pre-corrected directly within the author-provided expression matrix.
Modeling Strategy: Individual RIN scores are excluded as downstream linear model covariates
to prevent double-adjusting for RNA integrity.
"""
print(qc_statement)

# Export text artifact
with open("rna_qc_rin_statement.txt", "w") as f:
    f.write(qc_statement)


# STEP 4: SUBSET EXPRESSION MATRIX & EXPORT CSV FILES
print("\n=== STEP 4: Exporting Cleaned CSV Artifacts ===")

# Identify matching sample columns in the FPKM matrix
hip_profile_ids = hip_target["rnaseq_profile_id"].astype(str).tolist()
valid_cols = ["gene_id"] + [
    c for c in fpkm_norm.columns if str(c) in hip_profile_ids
]

hip_fpkm_norm = fpkm_norm[valid_cols].copy()

# Save metadata and expression CSVs
hip_target.to_csv("final_hippocampal_rna_master_metadata.csv", index=False)
hip_fpkm_norm.to_csv("final_hippocampal_fpkm_normalized.csv", index=False)

# Build and export summary metrics table
summary_results = [
    {
        "Metric": "Total Hippocampal Samples (N)",
        "Category": "Sample Overview",
        "Value": len(hip_samples),
    },
    {
        "Metric": "Unique Donors in HIP",
        "Category": "Sample Overview",
        "Value": hip_samples["donor_id"].nunique(),
    },
    {
        "Metric": "Pathology-positive dementia Count",
        "Category": "Group Breakdown",
        "Value": pathology_count,
    },
    {
        "Metric": "Resilience-like Count",
        "Category": "Group Breakdown",
        "Value": resilience_count,
    },
    {
        "Metric": "Primary Group Missingness",
        "Category": "Quality Control",
        "Value": group_missingness,
    },
    {
        "Metric": "Total Genes Measured",
        "Category": "Expression Matrix",
        "Value": hip_fpkm_norm.shape[0],
    },
    {
        "Metric": "Total Expression Samples",
        "Category": "Expression Matrix",
        "Value": hip_fpkm_norm.shape[1] - 1,
    },
]

summary_df = pd.DataFrame(summary_results)
summary_df.to_csv("final_hippocampal_qc_summary.csv", index=False)

print("Saved CSV Files:")
print(" - final_hippocampal_rna_master_metadata.csv")
print(" - final_hippocampal_fpkm_normalized.csv")
print(" - final_hippocampal_qc_summary.csv")


# STEP 5: RUN PCA & EXPORT VECTOR PDF FIGURE
print("\n=== STEP 5: Computing PCA & Exporting PDF Figure ===")

# Set gene_id as matrix index
expr_matrix = hip_fpkm_norm.set_index("gene_id")

# Filter for top 1,000 most variable genes
gene_vars = expr_matrix.var(axis=1)
top_genes = gene_vars.nlargest(1000).index
expr_top = expr_matrix.loc[top_genes].T  # Transpose: Samples as rows

# Log2 transformation (log2(FPKM + 1))

# Fit PCA
pca = PCA(n_components=2)
pca_coords = pca.fit_transform(expr_log)
var_exp = pca.explained_variance_ratio_ * 100

# Assemble PCA DataFrame
pca_df = pd.DataFrame(
    pca_coords, columns=["PC1", "PC2"], index=expr_log.index
)

# Fix: Convert data types to match for merging
if hip_target["rnaseq_profile_id"].dtype == 'int64':
    pca_df["rnaseq_profile_id"] = pca_df.index.astype(int)
else:
    pca_df["rnaseq_profile_id"] = pca_df.index.astype(str)

# Merge PCA coordinates with metadata
pca_merged = pca_df.merge(
    hip_target[["rnaseq_profile_id", group_col]],
    on="rnaseq_profile_id",
    how="left",
)

# Render Plot
plt.figure(figsize=(8, 6))

colors = {
    "Pathology-positive dementia": "#d62728",
    "Resilience-like": "#2ca02c",
}

for group_name, group_data in pca_merged.groupby(group_col):
    color = colors.get(group_name, "#1f77b4")
    plt.scatter(
        group_data["PC1"],
        group_data["PC2"],
        label=str(group_name),
        color=color,
        alpha=0.85,
        s=70,
        edgecolors="k",
        linewidth=0.5,
    )

plt.xlabel(f"PC1 ({var_exp[0]:.1f}% Variance Explained)")
plt.ylabel(f"PC2 ({var_exp[1]:.1f}% Variance Explained)")
plt.title(
    "Hippocampal RNA-Seq PCA\n(RIN-Corrected Normalized FPKM, Top 1,000 Genes)"
)
plt.legend(title="Group", loc="best")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

# Save vector PDF plot
pdf_filename = "final_hippocampal_rna_pca_by_group.pdf"
plt.savefig(pdf_filename, format="pdf", bbox_inches="tight")
plt.close()

print(f"PCA plot successfully saved as PDF: '{pdf_filename}'")
print("\n=== PIPELINE PROCESSING COMPLETE ===")

=== STEP 1: Loading RIN-Corrected FPKM Matrix ===
Loaded expression matrix: 19230 Genes x 376 Samples

=== STEP 2: Loading Sample and Donor Metadata ===
Total Hippocampal Samples Identified: 94
Unique Donors in HIP: 94

=== STEP 3: Merging Donor Variables & Filtering Cohort ===
Cohort Sample Counts by Group:
primary_group
Pathology-positive dementia    29
Resilience-like                19
Name: count, dtype: int64

=== STEP 3.5: Logging RIN QC Statement ===
=== HIPPOCAMPAL RNA-SEQ QUALITY CONTROL (RIN) STATEMENT ===
Expression Matrix Used: GSE104687_RIN_corrected_FPKM.csv.gz
RIN Adjustment Status: Pre-corrected directly within the author-provided expression matrix.
Modeling Strategy: Individual RIN scores are excluded as downstream linear model covariates
to prevent double-adjusting for RNA integrity.


=== STEP 4: Exporting Cleaned CSV Artifacts ===
Saved CSV Files:
 - final_hippocampal_rna_master_metadata.csv
 - final_hippocampal_fpkm_normalized.csv
 - final_hippocampal_qc_summary.cs